In [ ]:
import numpy as np
import scipy.integrate
from perfetto.trace_processor import TraceProcessor
import pandas as pd
from typing import Dict, Any, Optional

In [ ]:
class LBO(object):
    GC_SLICE_SQL_LIKE = "% % GC"

    def __init__(self, trace_path: str):
        self.tp = TraceProcessor(trace=trace_path)
        self.__create_gc_slices_view()
    
    def query(self, q: str):
        return self.tp.query(q)
    
    def query_pandas(self, q: str) -> pd.DataFrame:
        return self.query(q).as_pandas_dataframe()

    def query_single(self, q: str) -> Dict[str, Any]:
        return next(self.query(q)).__dict__

    def execute(self, q: str):
        self.query(q)

    def __create_gc_slices_view(self):
        self.execute("""
            CREATE VIEW gc_slice AS
            SELECT
                slice.id AS slice_id,
                slice.ts AS slice_start,
                slice.dur AS slice_dur,
                slice.ts + slice.dur AS slice_end,
                slice.name AS slice_name,
                thread.utid,
                thread.name AS thread_name,
                process.upid,
                process.name AS process_name
            FROM slice
            INNER JOIN thread_track on thread_track.id = slice.track_id
            INNER JOIN thread on thread_track.utid = thread.utid
            INNER JOIN process on thread.upid = process.upid
            WHERE slice.name LIKE '{}'
        """.format(LBO.GC_SLICE_SQL_LIKE))

    # XXX the SoC can be heterogenous so adding up CPU times across cores is not quite sound
    def get_gc_slices(self, process: Optional[str] = None) -> pd.DataFrame:
        # Enable scheduling and atrace_categories: "dalvik"
        return self.query_pandas("""
            SELECT *
            FROM gc_slice
            {}
        """.format("WHERE process_name = '{}'".format(process) if process else ""))

    def get_gc_slices_cpu_time(self, process: Optional[str] = None) -> pd.DataFrame:
        # Enable scheduling and atrace_categories: "dalvik"
        return self.query_pandas("""
            SELECT
                gc_slice.*,
                SUM(
                    IIF(thread_state.ts + thread_state.dur <= gc_slice.slice_end, thread_state.ts + thread_state.dur, gc_slice.slice_end) -
                    IIF(thread_state.ts >= gc_slice.slice_start, thread_state.ts, gc_slice.slice_start)
                ) AS cpu_time
            FROM gc_slice
            INNER JOIN thread_state ON thread_state.utid = gc_slice.utid
            WHERE
                thread_state.state = 'Running' AND
                (
                    (thread_state.ts >= gc_slice.slice_start AND thread_state.ts <= gc_slice.slice_end) OR
                    (thread_state.ts + thread_state.dur >= gc_slice.slice_start AND thread_state.ts + thread_state.dur <= gc_slice.slice_end)
                )
                {}
            GROUP BY gc_slice.slice_id
        """.format("AND process_name = '{}'".format(process) if process else ""))

    def get_process_cpu_time(self, process: str) -> float:
        return self.query_single("""
            SELECT SUM(dur) AS cpu_time
            FROM thread_state
            INNER JOIN thread ON thread.utid = thread_state.utid
            INNER JOIN process on process.upid = thread.upid
            WHERE process.name = '{}' AND thread_state.state = 'Running'
        """.format(process))["cpu_time"]
    
    def cpufreq(self):
        return self.tp.query("""
            SELECT ts, value, cpu
            FROM counter
            INNER JOIN cpu_counter_track ON counter.track_id=cpu_counter_track.id
            WHERE cpu_counter_track.name='cpufreq'
        """).as_pandas_dataframe()

    def integrate_mem(self, process: str, mem_type: str = "mem.rss.anon") -> int:
        # Enable ftrace_events: "kmem/rss_stat" in Perfetto
        # mem_type can be "mem.rss.anon", "mem.rss.file", "mem.rss.shmem", "mem.swap"
        df = self.tp.query("""
            SELECT ts, value, process_counter_track.name, process.name
            FROM counter
            INNER JOIN process_counter_track on process_counter_track.id = counter.track_id
            INNER JOIN process on process.upid = process_counter_track.upid
            WHERE process_counter_track.name = '{}' AND process.name = '{}'
            ORDER BY ts ASC
        """.format(mem_type, process)).as_pandas_dataframe()
        print(df)
        # value is in byte
        # ts is in nanosecond
        return scipy.integrate.trapezoid(df["value"], df["ts"]) / 1e9 / 1024 / 1024 # MB * second

In [ ]:
lbo = LBO("example-SS.perfetto-trace")

In [ ]:
lbo.get_gc_slices("com.google.android.apps.maps")

In [ ]:
lbo.get_gc_slices_cpu_time("com.google.android.apps.maps")

In [ ]:
lbo.get_process_cpu_time("com.google.android.apps.maps")

In [ ]:
lbo.get_gc_slices_cpu_time("com.google.android.apps.maps")["cpu_time"].sum()/lbo.get_process_cpu_time("com.google.android.apps.maps")